# 05 - Testing input improvements
Two ideas to improve the planning model. We only keep an idea if it helps on **seasons the model has never seen**, because that was our biggest weakness.

- **Fewer weather columns:** temperature, apparent temperature and dew point are 95-99% copies of each other. Keep 4: temperature, precipitation, snowfall, wind speed.
- **Typical-ridership column:** the average ridership for the same route, hour and weekday/weekend, calculated from training data only. For training rows the row's own value is left out of the average, so the model cannot learn from a number that contains the answer.

Same Random Forest settings every time. Test set is not used.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from utils import TARGET, best_forest, load_data, make_pipeline, planning_features, split_by_days

sns.set_theme(style="whitegrid")
df = load_data()
train, val, test = split_by_days(df)
planning = planning_features(df, fewer_weather=False)
weather_all = ["temperature_2m", "apparent_temperature", "relative_humidity_2m", "dew_point_2m", "precipitation",
               "snowfall", "pressure_msl", "cloud_cover", "wind_speed_10m", "wind_gusts_10m"]
weather_keep = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]
fewer_weather = [c for c in planning if c not in weather_all or c in weather_keep]
print(len(df), "rows;", len(planning), "planning features;", len(fewer_weather), "with fewer weather columns")

64512 rows; 26 planning features; 20 with fewer weather columns


In [2]:
versions = {
    "A: current features": (planning, False),
    "B: fewer weather columns": (fewer_weather, False),
    "C: + typical ridership": (planning, True),
    "D: both": (fewer_weather, True),
}

def fit_score(train_part, test_part, features, typical):
    pipe = make_pipeline(best_forest(), features, typical=typical).fit(train_part[features], train_part[TARGET])
    return mean_absolute_error(test_part[TARGET], pipe.predict(test_part[features]))

## Leave one window out (8 windows)
Each of the 8 two-week windows is held out in turn. Lower is better.

In [3]:
rows = []
for w in sorted(df["window_id"].unique()):
    tr, te = df[df["window_id"] != w], df[df["window_id"] == w]
    row = {"held-out window": w}
    for name, (features, typical) in versions.items():
        row[name] = fit_score(tr, te, features, typical)
    rows.append(row)
loo = pd.DataFrame(rows).set_index("held-out window")
loo.loc["mean"] = loo.mean()
loo.round(1)

,A: current features,B: fewer weather columns,C: + typical ridership,D: both
held-out window,,,,
0,49.1,47.1,50.1,49.3
1,44.8,44.7,49.0,49.1
2,47.2,47.1,51.2,52.2
3,44.2,45.9,49.4,51.3
4,69.9,69.7,66.1,65.1
5,54.0,49.9,53.1,49.7
6,58.7,54.2,61.5,60.2
7,63.4,63.1,65.6,65.8
mean,53.9,52.7,55.7,55.4


In [4]:
vs_a = loo.drop(index="mean").sub(loo.drop(index="mean")["A: current features"], axis=0)
summary = pd.DataFrame({"mean MAE": loo.loc["mean"], "windows better than A": (vs_a < 0).sum()})
summary.round(1)

,mean MAE,windows better than A
A: current features,53.9,0
B: fewer weather columns,52.7,7
C: + typical ridership,55.7,2
D: both,55.4,2


## Normal time split (train -> validation)
A second view, closer to how the final model is trained.

In [5]:
time_split = pd.Series({name: fit_score(train, val, f, t) for name, (f, t) in versions.items()}, name="val MAE")
time_split.round(1)

A: current features         59.7
B: fewer weather columns    55.3
C: + typical ridership      63.3
D: both                     59.2
Name: val MAE, dtype: float64

## Decision
- **Adopt B (fewer weather columns).** Lower error on 7 of 8 held-out windows (52.7 vs 53.9 on average) and on the normal time split (55.3 vs 59.7). The six dropped columns (`apparent_temperature`, `relative_humidity_2m`, `dew_point_2m`, `pressure_msl`, `cloud_cover`, `wind_gusts_10m`) are in `utils.WEATHER_DROPPED`.
- **Reject C (typical-ridership column).** It is worse on both views (55.7 and 63.3). The trees already learn the route-hour average from `bus_route`, `hour` and `is_weekend`, so the extra column mostly adds noise.
- For context, with the earlier 4-window dataset the held-out-season error was about 73; with 8 windows it is about 54. The datasets differ (24 routes instead of 16, different windows), so this is not a strict comparison, but more seasons clearly helped.